In [23]:
import tensorflow as tf
from keras import layers,models,preprocessing
import numpy as np
import os
import matplotlib.pyplot as plt


In [24]:
DATA_DIR="celebA"
IMG_HEIGHT=64
IMG_WIDTH=64
BATCH_SIZE=128
BUFFER_SIZE=60000

In [25]:
def load_image(image_path):
    image=tf.io.read_file(image_path)
    image=tf.image.decode_jpeg(image, channels=3)
    image=tf.image.resize(image, [IMG_HEIGHT,IMG_WIDTH])
    image=(image-127.5)/127.5
    return image

In [26]:
def load_dataset(data_dir):
    image_paths=[os.path.join(data_dir,img) for img in os.listdir(data_dir)]
    image_dataset=tf.data.Dataset.from_tensor_slices(image_paths)
    image_dataset=image_dataset.map(load_image,num_parallel_calls=tf.data.experimental.AUTOTUNE)
    image_dataset=image_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.experimental.AUTOTUNE)
    return image_dataset

In [27]:
train_dataset=load_dataset(DATA_DIR)

In [32]:
def build_generator():
    model = models.Sequential()
    model.add(layers.Dense(8 * 8 * 256, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((8, 8, 256)))
    assert model.output_shape == (None, 8, 8, 256) # Убедитесь, что выходная форма такая
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same',
    use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    assert model.output_shape == (None, 16, 16, 128)
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same',
    use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    assert model.output_shape == (None, 32, 32, 64)
    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False,activation='tanh'))
    assert model.output_shape == (None, 64, 64, 3)
    return model


In [36]:
def build_discriminator():
    model = models.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[64, 64, 3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    model.add(layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))
    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))
    return model

In [38]:
generator=build_generator()
discriminator=build_discriminator()
cross_entropy =tf._losses.BinaryCrossentropy(from_logits=True)

In [39]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [40]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [42]:
generator_optimizer = tf._optimizers.Adam(1e-4)
discriminator_optimizer = tf._optimizers.Adam(1e-4)